# uMLIP 12종 벤치마크 v1 — 앵커 4-point (HOST/HEO) + XRD (RunPod 멀티 GPU)

MODEL_BENCHMARK_SPEC의 관문 0(레지스트리)·관문 1(앵커 방향)을 **DFT 없이 지금** 실행한다.
정답지(DFT CORE/FRAMES)는 아직 없으므로 절대 오차(MAE)는 없고, 있는 것은:
(a) 실험 사실 2개 — HOST는 전이(V≈3.1 V), HEO는 지연 — 에 대한 모델별 방향 시험,
(b) 12모델의 dE(x)·x\*·V_seg 상호 비교, (c) XRD 지문(세 번째 상 판독기, XRD_SPEC), (d) 속도.

| 원칙 (스펙 §10) | 구현 |
|---|---|
| 같은 시작구조·조건·GPU | SQS 캐시 공유(shared_base) + 전 모델 ASE FIRE + FrechetCellFilter, fmax/스텝 동일 |
| E_Na 혼용 금지 | 모델별 자기 bcc Na |
| 화폐 불일치 모델 제외 금지 | orb_omat 등 대조군으로 결과 표 유지 |
| 채점 정의 불변 | 스크리닝 workdir(heo_v2/v4)에는 아무것도 쓰지 않음 |
| 앵커 파인튜닝 금지 | 학습 없음 |

**주의 — 프로덕션과의 경로 차이**: v4 스크리닝은 orb를 torch-sim 배치로 이완했다. 벤치마크는 12모델
균일성을 위해 전부 ASE 경로다. 셀 8이 v4 앵커 결과와 orb-ASE를 대조해 그 차이를 정량화한다.

```
0 필수 모듈 설치(메인 env)   1 CONFIG   1b 워치독   2 INSTALL(모델별 venv)   3 파일·어댑터 확인
4 shared_base + 앵커 SQS 선생성   5 런처+스케줄러(블로킹)   6 상태 요약(배리어)
7 교차 집계 → bench_summary.csv   8 orb ASE↔torch-sim 일관성
8b XRD on 기존 v4 결과   8c XRD 워터폴 pptx   9 리포트 md
```


## 셀 0 — 필수 모듈 설치 (메인 jupyter env)
모델별 torch/venv는 **셀 2**가 만든다 — 여기는 노트북·러너·SQS 생성이 직접 쓰는 공통 모듈만.

In [ ]:
# ============ CELL 0: base modules (main env) ============
# API 키는 절대 여기 쓰지 말 것 -- 터미널 export만 (MP_API_KEY / HF_TOKEN / RUNPOD_API_KEY).
import sys, subprocess
BASE_PKGS = ["pandas", "numpy", "scipy", "pymatgen", "ase", "pyyaml", "icet",
             "python-pptx", "matplotlib", "huggingface_hub", "papermill", "uv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *BASE_PKGS])
import importlib
missing = []
for m in ("pandas", "numpy", "scipy", "pymatgen", "ase", "yaml", "icet", "pptx",
          "matplotlib", "huggingface_hub"):
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append(f"{m}: {type(e).__name__}")
print("base modules OK" if not missing else f"IMPORT FAIL -> {missing}")


## 셀 0b — 사용자 설정 (직접 대입)
셀 1은 전부 `setdefault`라 **여기서 직접 대입한 값이 항상 이긴다** (커널에 낡은 값이 박혀 있어도).
바꾼 뒤에는 이 셀부터 다시 실행; 워치독이 이미 떠 있으면 터미널에서 `pkill -f pod_watchdog.py` 후 셀 1b 재실행.

In [ ]:
# ============ CELL 0b: user overrides (direct assignment -- setdefault 아님!) ============
import os
os.environ["HEO_TERMINATE_MODE"] = "stop"    # none | stop | terminate
os.environ["HEO_BENCH_MODELS"]   = ""        # "" = 12종 전부, 스모크는 "orb_mpa"
os.environ["HEO_BENCH_TOPN"]     = "30"      # results_v3.csv top-N 재계산 규모
os.environ["HEO_BENCH_EHULL"]    = "0"       # 0 = 경쟁상(hull) 생략 -- top-N 자체는 계산됨
os.environ["HEO_EQUFLASH_CKPT"]  = "/workspace/ckpts/equflashv2-45m-oam.pt"   # M09 Figshare ckpt
# os.environ["HEO_ESEN_MODEL"]   = "uma-m-1p1"   # M05 대체 유닛 (HF_TOKEN은 터미널 export -- 키를 노트북에 쓰지 말 것)
# os.environ["HEO_XRD_SCOPE"]    = "top30"       # pilot | top30 | top500 (셀 8b)
print({k: v for k, v in sorted(os.environ.items()) if k.startswith("HEO_")})


## 셀 1 — 설정

In [ ]:
# ============ CELL 1: CONFIG ============
import os, sys, json, time, glob, subprocess
from pathlib import Path
import pandas as pd

os.environ.setdefault("HEO_BENCH_ROOT",   "/workspace/heo_bench")
BENCH  = os.environ["HEO_BENCH_ROOT"]
SHARED = f"{BENCH}/shared_base"          # structures/sqs ONLY -- the one thing every model shares
RUNS   = f"{BENCH}/runs"                 # one workdir per model (invariant 6: one currency per workdir)
VENVS  = f"{BENCH}/venvs"
os.environ.setdefault("HEO_V2_WORKDIR",   "/workspace/heo_v2")    # v3 run (SQS source, XRD structures)
os.environ.setdefault("HEO_V4_WORKDIR",   "/workspace/heo_v4")    # v4 run (XRD structures, orb comparison)
os.environ.setdefault("HEO_V4_RESULTS",   "/workspace/heo_v4/results/anchors_x4.csv")
os.environ.setdefault("HEO_BASE_RESULTS", "/workspace/heo_v2/results/results_v3.csv")
os.environ.setdefault("HEO_BENCH_MODELS", "")        # "" = all 12; smoke test: "orb_mpa" or "mock"
os.environ.setdefault("HEO_XRD_SCOPE",    "top30")   # pilot | top30 | top500  (cell 8b)
os.environ.setdefault("HEO_TERMINATE_MODE", "none")
MAX_HOURS = float(os.environ.get("HEO_MAX_HOURS", 36)); IDLE_MIN = float(os.environ.get("HEO_IDLE_MIN", 180))
WORKDIR = BENCH                                       # the watchdog guards the whole bench tree

for d in (BENCH, SHARED, f"{SHARED}/structures/sqs", RUNS, VENVS,
          f"{BENCH}/results", f"{BENCH}/logs", f"{BENCH}/model_ckpts"):
    os.makedirs(d, exist_ok=True)
Path(BENCH, "results", "DONE").unlink(missing_ok=True)

# heo_worker is imported by THIS kernel in cell 4 (SQS pre-generation): it must see the shared
# workdir and a non-mock mode, or _sqs_worker would write RANDOM placements instead of icet SQS.
os.environ["HEO_WORKDIR"] = SHARED
if os.path.isdir(os.environ["HEO_V2_WORKDIR"]):
    os.environ["HEO_BASE_WORKDIR"] = os.environ["HEO_V2_WORKDIR"]   # reuse v3 anchor SQS if present
else:
    os.environ.pop("HEO_BASE_WORKDIR", None)
    print(f"[warn] {os.environ['HEO_V2_WORKDIR']} not found -> anchor SQS will be generated fresh (icet)")
os.environ["HEO_MODE"] = "orb"          # only affects worker IMPORT here; no relaxation runs in this kernel

N_GPUS = 0
try:
    import torch; N_GPUS = torch.cuda.device_count()
except Exception:
    try:
        _o = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
        N_GPUS = sum(1 for l in _o.splitlines() if l.startswith("GPU "))
    except (FileNotFoundError, OSError):
        N_GPUS = 0
N_GPUS = int(os.environ.get("HEO_NGPU", 0)) or N_GPUS or 1

sys.path.insert(0, os.getcwd())
import bench_models
MODELS = [m for m in os.environ["HEO_BENCH_MODELS"].split(",") if m] or list(bench_models.DEFAULT_MODELS)
assert all(m in bench_models.ADAPTERS for m in MODELS), f"unknown adapter in HEO_BENCH_MODELS: {MODELS}"

class _Tee:
    _heo_tee = True
    def __init__(self, stream, path): self._s = stream; self._f = open(path, "a", buffering=1)
    def write(self, x): self._s.write(x); self._f.write(x); return len(x)
    def flush(self): self._s.flush(); self._f.flush()
    def __getattr__(self, a): return getattr(self._s, a)
_LOG = f"{BENCH}/results/notebook_stdout.log"
if not getattr(sys.stdout, "_heo_tee", False):
    sys.stdout = _Tee(sys.stdout, _LOG); sys.stderr = _Tee(sys.stderr, _LOG)
print(f"\n===== bench run started {time.strftime('%Y-%m-%d %H:%M:%S')} -> also logging to {_LOG} =====")
print(f"GPUs={N_GPUS} | bench={BENCH} | models({len(MODELS)}): {MODELS}")
print(f"terminate={os.environ['HEO_TERMINATE_MODE']} | max {MAX_HOURS} h | idle {IDLE_MIN} min")


## 셀 1b — 팟 과금 방지 워치독 (v4 셀 1b 그대로)

In [ ]:
# ============ CELL 1b: pod cost guard -- detached watchdog (identical to v4) ============
import textwrap
WATCHDOG_SRC = textwrap.dedent(r"""
    import os, sys, time, subprocess, urllib.request
    from pathlib import Path
    WORKDIR = Path(sys.argv[1]); MAX_H = float(sys.argv[2]); IDLE_MIN = float(sys.argv[3]); MODE = sys.argv[4]
    LOG = open(WORKDIR / "watchdog.log", "a", buffering=1)
    def say(m): LOG.write(f"[{time.strftime('%H:%M:%S')}] {m}\n")

    def pod_terminate(mode):
        pod = os.environ.get("RUNPOD_POD_ID")
        if not pod: return False, "RUNPOD_POD_ID not set"
        cmds = ([["runpodctl","pod","delete",pod],["runpodctl","remove","pod",pod]] if mode == "delete"
                else [["runpodctl","pod","stop",pod],["runpodctl","stop","pod",pod]])
        for c in cmds:
            try:
                if subprocess.run(c, capture_output=True).returncode == 0: return True, " ".join(c)
            except Exception as e: say(f"  {c[0]} unusable: {e!r}")
        key = os.environ.get("RUNPOD_API_KEY")
        if not key: return False, "runpodctl failed and RUNPOD_API_KEY not set"
        url = f"https://rest.runpod.io/v1/pods/{pod}" + ("" if mode == "delete" else "/stop")
        req = urllib.request.Request(url, method="DELETE" if mode == "delete" else "POST",
                                     headers={"Authorization": f"Bearer {key}"})
        try:
            urllib.request.urlopen(req, timeout=20); return True, "REST " + url
        except Exception as e: return False, f"REST failed: {e!r}"

    def gpus_busy():
        try:
            out = subprocess.run(["nvidia-smi","--query-gpu=utilization.gpu","--format=csv,noheader,nounits"],
                                 capture_output=True, text=True, timeout=30).stdout
            return any(int(x) > 5 for x in out.split() if x.strip().isdigit())
        except Exception:
            return False

    def snapshot():
        s = []
        for p in WORKDIR.rglob("*"):
            if p.is_file() and p.name != "watchdog.log":
                try: s.append((str(p), p.stat().st_size, int(p.stat().st_mtime)))
                except OSError: pass
        return hash(tuple(sorted(s)))

    def fire(mode, why):
        if mode == "none":
            say(f"LOG-ONLY - would terminate now: {why}")
            return
        say(f"TERMINATING ({mode}) - {why}")
        ok, how = pod_terminate(mode)
        say(f"  -> {'OK ' + how if ok else 'FAILED: ' + how}")
        sys.exit(0 if ok else 1)

    say(f"watchdog up | max {MAX_H} h | idle {IDLE_MIN} min | mode {MODE} | pod {os.environ.get('RUNPOD_POD_ID')}")
    t0 = time.time(); last_sig = snapshot(); last_change = time.time(); done_at = None
    while True:
        time.sleep(60)
        if (WORKDIR / "results" / "DONE").exists():
            done_at = done_at or time.time()
            if time.time() - done_at > 300:
                fire(MODE, "DONE written, pod still up after 5 min"); done_at += 21600
            continue
        if time.time() - t0 > MAX_H * 3600:
            fire(MODE, f"hard deadline {MAX_H} h"); t0 += 21600
        sig = snapshot()
        if sig != last_sig: last_sig, last_change = sig, time.time(); continue
        idle_s = time.time() - last_change
        if idle_s > IDLE_MIN * 60:
            if gpus_busy():
                say(f"  files idle {idle_s/60:.0f} min but GPUs busy - holding"); last_change = time.time()
            else:
                fire(MODE, f"no file activity for {idle_s/60:.0f} min and GPUs idle")
                last_change = time.time()
""")
_wd_mode = os.environ["HEO_TERMINATE_MODE"]
if os.environ.get("HEO_WATCHDOG", "1") == "1" and os.environ.get("RUNPOD_POD_ID"):
    _wd = Path(BENCH, "pod_watchdog.py"); _wd.write_text(WATCHDOG_SRC)
    _p = subprocess.Popen([sys.executable, str(_wd), BENCH, str(MAX_HOURS), str(IDLE_MIN), _wd_mode],
                          env=dict(os.environ), start_new_session=True,
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"pod watchdog started (pid {_p.pid}) -> max {MAX_HOURS} h, idle {IDLE_MIN} min, "
          f"mode {_wd_mode}, log {BENCH}/watchdog.log")
    if _wd_mode == "none":
        print("  terminate DISABLED -> log-only watchdog; the pod BILLS until you stop it yourself")
elif not os.environ.get("RUNPOD_POD_ID"):
    print("pod watchdog: RUNPOD_POD_ID not set -> not on a RunPod pod, nothing to guard")
else:
    print("pod watchdog DISABLED (HEO_WATCHDOG=0)")


## 셀 2 — 설치 (베이스 + 모델별 venv)

의존성 충돌 격리: 기본은 `--system-site-packages` venv(공유 torch + 모델 패키지만),
`isolated`(M06/M09/M10/M11)는 자체 torch/python 포함 완전 격리(uv 필요, 없으면 pip로 uv 설치).
**설치 실패는 기록 후 계속** — 스펙 §10-6, 실패 모델도 결과 표에 '실행 불가'로 남는다.

In [ ]:
# ============ CELL 2: INSTALL ============
import importlib.util, shutil
from importlib.metadata import version as _ver, PackageNotFoundError

BASE_PKGS = {"pymatgen": "pymatgen", "ase": "ase", "scipy": "scipy", "yaml": "pyyaml",
             "icet": "icet", "pptx": "python-pptx", "matplotlib": "matplotlib",
             "huggingface_hub": "huggingface_hub", "papermill": "papermill"}
missing = [p for m, p in BASE_PKGS.items() if importlib.util.find_spec(m) is None]
if missing:
    print("base env installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

INSTALL_STATUS = {}
def _venv_python(key):
    p = f"{VENVS}/{key}/bin/python"
    return p if os.path.exists(p) else None

def _pip(py, pkgs):
    """pip inside the venv.  uv-created venvs ship WITHOUT pip (python -m pip dies instantly,
    which is what killed the tece install on 09-17) -> fall back to `uv pip install --python`."""
    try:
        subprocess.check_call([py, "-m", "pip", "install", "-q", *pkgs])
    except (subprocess.CalledProcessError, FileNotFoundError):
        uv = shutil.which("uv") or f"{os.path.dirname(sys.executable)}/uv"
        subprocess.check_call([uv, "pip", "install", "-q", "--python", py, *pkgs])

def _make_venv(key, ad):
    vdir = f"{VENVS}/{key}"
    if _venv_python(key):
        return "exists"
    if ad.get("isolated"):
        uv = shutil.which("uv")
        if uv is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
            uv = shutil.which("uv") or f"{os.path.dirname(sys.executable)}/uv"
        cmd = [uv, "venv", vdir] + (["--python", ad["python"]] if ad.get("python") else ["--system-site-packages"])
        subprocess.check_call(cmd)
        if ad.get("torch"):
            _pip(f"{vdir}/bin/python", [ad["torch"]])
    else:
        subprocess.check_call([sys.executable, "-m", "venv", "--system-site-packages", vdir])
    return "created"

for key in MODELS:
    ad = bench_models.ADAPTERS[key]
    t0 = time.time()
    try:
        how = _make_venv(key, ad)
        py = _venv_python(key)
        for step in ad.get("pip_steps", []):     # multi-stage installs (own index-url per step, e.g. M06)
            _pip(py, step)
        if ad["pip"]:
            _pip(py, ad["pip"])
        chk = subprocess.run([py, "-c",
                              f"import sys; sys.path.insert(0, {os.getcwd()!r}); "
                              f"import bench_models; print(bench_models.check_import({key!r}))"],
                             capture_output=True, text=True, timeout=600)
        status = chk.stdout.strip() or ("FAIL " + chk.stderr.strip()[-200:])
        INSTALL_STATUS[key] = {"venv": how, "import": status, "t_s": round(time.time() - t0, 1)}
    except Exception as e:
        INSTALL_STATUS[key] = {"venv": "ERROR", "import": f"{type(e).__name__}: {str(e)[:200]}",
                               "t_s": round(time.time() - t0, 1)}
    print(f"  {key:<18} {INSTALL_STATUS[key]['venv']:<8} {INSTALL_STATUS[key]['import']}")

json.dump(INSTALL_STATUS, open(f"{BENCH}/results/pkg_versions_bench.json", "w"), indent=1)
n_ok = sum(1 for v in INSTALL_STATUS.values() if v["import"].startswith("ok"))
print(f"\n{n_ok}/{len(MODELS)} adapters importable -- failures stay in the table as '실행 불가' (spec 10-6)")


## 셀 3 — 파일·어댑터 확인

In [ ]:
# ============ CELL 3: files + adapter availability ============
import hashlib
for fn in ("heo_worker.py", "bench_models.py", "bench_anchor_runner.py", "xrd_tools.py"):
    assert os.path.exists(fn), f"{fn} missing next to this notebook -- upload it"
    print(f"{fn:<24} md5 {hashlib.md5(open(fn,'rb').read()).hexdigest()}")
import importlib, bench_models as _bm; importlib.reload(_bm); bench_models = _bm
print(f"\nadapters ({len(bench_models.ADAPTERS)-1} models + mock):")
for k in MODELS:
    a = bench_models.ADAPTERS[k]
    imp = INSTALL_STATUS.get(k, {}).get("import", "?")
    print(f"  {a['model_id']:>4} {k:<18} isolated={str(a['isolated']):<5} import={imp:<24} {a['currency']}")


## 셀 4 — shared_base 구성 + 앵커 SQS 선생성

모든 모델이 **같은 SQS 셀에서 출발**해야 한다(스펙 §10-1). v3(heo_v2) 캐시가 있으면 복사,
없으면 icet로 여기서 생성한다. 러너는 SQS가 없으면 중단하도록 되어 있다(모의 랜덤 배치 유입 방지).

In [ ]:
# ============ CELL 4: shared_base + anchor SQS pre-generation ============
import shutil
import heo_worker as W
assert W.MODE != "mock", "worker imported in mock mode -> SQS pre-generation would be RANDOM, not icet"
print("worker:", W.__file__, "| model-independent structure code only is used in this kernel")
W._validate_templates(verbose=False)
print("anchors:", {k: sum(v.values()) for k, v in W.ANCHORS.items()}, "| Ti03 permanently removed (2026-09-03)")

for aid, counts in W.ANCHORS.items():
    dst = f"{SHARED}/structures/sqs/{aid}.json"
    if os.path.exists(dst):
        print(f"  {aid}: shared cache OK"); continue
    src = W.sqs_path(aid)                       # own (=shared) first, then HEO_BASE_WORKDIR (= heo_v2)
    if os.path.exists(src) and os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst); print(f"  {aid}: copied from {src}")
    else:
        t0 = time.time(); cid, how = W._sqs_worker((aid, counts))
        print(f"  {aid}: generated ({how}, {time.time()-t0:.0f}s)")
        assert "fallback" not in how, f"{aid}: SQS fell back to random -> fix icet before benchmarking"
for aid in W.ANCHORS:
    assert os.path.exists(f"{SHARED}/structures/sqs/{aid}.json")
print("anchor SQS ready:", sorted(os.listdir(f"{SHARED}/structures/sqs"))[:6], "...")

# ---- Ehull extension inputs (v4 cell-8 reuse): top-N targets + their SQS + competitor cache ----
# The hull is in-model on BOTH sides (each model relaxes the same MP competitor structures itself),
# exactly as production; what is shared here is only structures, never energies.
TOPN = int(os.environ.get("HEO_BENCH_TOPN", 30))
BR = os.environ["HEO_BASE_RESULTS"]
if TOPN and os.path.exists(BR):
    br = pd.read_csv(BR, index_col="comp_id").dropna(subset=["dE_desod"])
    if {"score", "tier"} <= set(br.columns):
        br = br.sort_values(["tier", "score"])
    top = list(br.index[:TOPN])
    cm = None
    for cand in (f"{os.environ['HEO_V4_WORKDIR']}/results/compositions_master.csv",
                 f"{os.environ['HEO_V2_WORKDIR']}/results/compositions_master.csv"):
        if os.path.exists(cand):
            cm = pd.read_csv(cand).set_index("comp_id"); break
    assert cm is not None, "compositions_master.csv not found under heo_v4/heo_v2 -> mount the volumes"
    rows, n_new = [], 0
    for cid in top:
        if cid not in cm.index: continue
        rows.append(dict(comp_id=cid, site_counts=cm.loc[cid, "site_counts"],
                         sconf_per_atom_kB=float(cm.loc[cid, "sconf_per_atom_kB"])))
        dst = f"{SHARED}/structures/sqs/{cid}.json"
        if not os.path.exists(dst):
            src = W.sqs_path(cid)                 # finds the v3 cache through HEO_BASE_WORKDIR
            if os.path.exists(src) and os.path.abspath(src) != os.path.abspath(dst):
                shutil.copy(src, dst)
            else:
                _cid, how = W._sqs_worker((cid, json.loads(cm.loc[cid, "site_counts"])))
                assert "fallback" not in how, f"{cid}: SQS fell back to random -> fix icet"
            n_new += 1
    pd.DataFrame(rows).to_csv(f"{SHARED}/bench_targets.csv", index=False)
    print(f"ehull targets: {len(rows)} comps (SQS staged, {n_new} new)")
    os.makedirs(f"{SHARED}/hull_cache", exist_ok=True)
    hc_dst = f"{SHARED}/hull_cache/competitors.json"
    if not os.path.exists(hc_dst):
        for hc_src in (f"{os.environ['HEO_V2_WORKDIR']}/hull_cache/competitors.json",
                       f"{os.environ['HEO_V4_WORKDIR']}/hull_cache/competitors.json"):
            if os.path.exists(hc_src):
                shutil.copy(hc_src, hc_dst); print("competitor cache copied:", hc_src); break
    if not os.path.exists(hc_dst):
        print("[warn] competitors.json not found in heo_v2/heo_v4 -> runners will SKIP Ehull "
              "(structures cache comes from the v4 screening's cell 8)")
else:
    print("ehull extension disabled (HEO_BENCH_TOPN=0 or HEO_BASE_RESULTS missing)")


## 셀 5 — 런처 + 스케줄러 (블로킹)

모델 큐를 GPU에 라운드로빈 배정, 동시 실행 ≤ GPU 수. 러너는 체크포인트 재개가 되므로
비정상 종료 시 **1회 자동 재기동**. 이 셀은 전 모델이 끝날 때까지 블로킹한다(Run All 안전).

In [ ]:
# ============ CELL 5: launcher + scheduler (blocking barrier built in) ============
def venv_python(key):
    p = f"{VENVS}/{key}/bin/python"
    return p if os.path.exists(p) else sys.executable

def spawn(key, gpu, relaunched=False):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), HEO_MODE="mock",
               HEO_WORKDIR=f"{RUNS}/{key}", HEO_BASE_WORKDIR=SHARED,
               HEO_BENCH_ROOT=BENCH, HEO_BENCH_CKPTS=f"{BENCH}/model_ckpts")
    log = open(f"{BENCH}/logs/{key}.log", "a")
    p = subprocess.Popen([venv_python(key), "bench_anchor_runner.py", "--adapter", key],
                         env=env, stdout=log, stderr=subprocess.STDOUT)
    RUNNING[key] = dict(proc=p, gpu=gpu, relaunched=relaunched, t0=time.time())
    print(f"[launch] {key} -> GPU {gpu} (pid {p.pid}, {venv_python(key)})", flush=True)

QUEUE = [k for k in MODELS]
RUNNING, EXITCODES = {}, {}
free_gpus = list(range(N_GPUS))
t_status = 0
while QUEUE or RUNNING:
    while QUEUE and free_gpus:
        spawn(QUEUE.pop(0), free_gpus.pop(0))
    time.sleep(20)
    for key in list(RUNNING):
        info = RUNNING[key]; code = info["proc"].poll()
        if code is None: continue
        del RUNNING[key]
        if code != 0 and not info["relaunched"]:
            print(f"[relaunch-once] {key} exited {code} after {(time.time()-info['t0'])/60:.0f} min "
                  f"(checkpoints resume)", flush=True)
            spawn(key, info["gpu"], relaunched=True)
        else:
            free_gpus.append(info["gpu"]); EXITCODES[key] = code
            print(f"[done] {key} exit {code} ({(time.time()-info['t0'])/60:.0f} min)", flush=True)
    if time.time() - t_status > 300:
        t_status = time.time()
        for key, info in RUNNING.items():
            lines = [l for l in open(f"{BENCH}/logs/{key}.log").read().strip().split("\n") if l.strip()]
            print(f"  [{key} @GPU{info['gpu']} {(time.time()-info['t0'])/60:.0f}min] {lines[-1] if lines else '(no output)'}",
                  flush=True)
print("\nall models finished:", EXITCODES)


## 셀 6 — 상태 요약 (배리어)

In [ ]:
# ============ CELL 6: per-model status summary ============
rows = []
for key in MODELS:
    fp = f"{RUNS}/{key}/results/anchor_bench.json"
    if os.path.exists(fp):
        s = json.load(open(fp))
        rows.append(dict(adapter=key, model_id=s.get("model_id"), status=s.get("status"),
                         exit=EXITCODES.get(key, "?"), n_rows=s.get("n_rows"),
                         warn=len(s.get("warnings", []))))
    else:
        rows.append(dict(adapter=key, model_id="?", status="NO OUTPUT", exit=EXITCODES.get(key, "?")))
ST = pd.DataFrame(rows).set_index("adapter")
print(ST.to_string())
n_ok = int((ST.status == "ok").sum())
print(f"\n{n_ok}/{len(MODELS)} models completed clean; failures are DATA (spec 10-6), not stoppers.")
assert n_ok > 0, "no model produced results -> read the logs in " + f"{BENCH}/logs/"


## 셀 7 — 교차 집계 → `bench_summary.csv`

모델 × (관문 부울, dE(x) per 그룹, x\*, V_avg, gap 진단, XRD 방향/일치율, E_Na, 속도, 수렴률).
기준선(orb_mpa) 대비 8-값 dE 벡터(HOST/HEO × 4x)의 Spearman ρ와 부호 일치율도 계산한다.
**주의**: n=8이라 ρ는 참고치 — 우열 선언은 DFT 도착 후 짝지은 CI로만 (스펙 §5).

In [ ]:
# ============ CELL 7: cross-model aggregation ============
import numpy as np
from scipy.stats import spearmanr

DE_COLS = ["dE_pris", "dE_x081", "dE_x067", "dE_desod"]
rows = []
for key in MODELS:
    fp = f"{RUNS}/{key}/results/anchor_bench.json"
    r = dict(adapter=key)
    if not os.path.exists(fp):
        r["status"] = "NO OUTPUT"; rows.append(r); continue
    s = json.load(open(fp))
    r.update(model_id=s.get("model_id"), name=s.get("name"), currency=s.get("currency"),
             status=s.get("status"), E_Na=s.get("E_Na"),
             t_relax_median_s=s.get("t_relax_median_s"), conv_rate=s.get("conv_rate"))
    for g, v in (s.get("gates") or {}).items(): r[g] = v
    for g, v in (s.get("diag") or {}).items(): r[g] = v
    for grp, gv in (s.get("groups") or {}).items():
        for c in DE_COLS + ["x_star_class", "x_star_mid", "V_avg",
                            "V_seg_100_081", "V_seg_081_067", "V_seg_067_052", "n_phase_flip",
                            "dV_pct_desod", "dA_pct_desod", "dh_perp_pct_desod", "vm_strain_desod",
                            "dV_pct_x067", "dA_pct_x067", "dh_perp_pct_x067", "vm_strain_x067"]:
            if c in gv: r[f"{grp}_{c}"] = gv[c]
    x = s.get("xrd") or {}
    r.update(xrd_dir_ok_003=x.get("dir_ok_003"), xrd_dir_ok_110=x.get("dir_ok_110"),
             xrd_d003_bad_rows=x.get("d003_audit_bad_rows"), xrd_letter_agree=x.get("letter_agree_rate"))
    rows.append(r)
S = pd.DataFrame(rows).set_index("adapter")

BASELINE = "orb_mpa"
def _vec(row):
    return np.array([row.get(f"{g}_{c}", np.nan) for g in ("HOST", "HEO") for c in DE_COLS], dtype=float)
if BASELINE in S.index:
    v0 = _vec(S.loc[BASELINE])
    rho, sign = [], []
    for k in S.index:
        v = _vec(S.loc[k]); m = np.isfinite(v0) & np.isfinite(v)
        rho.append(spearmanr(v0[m], v[m]).statistic if m.sum() >= 3 else np.nan)
        sign.append(float((np.sign(v0[m]) == np.sign(v[m])).mean()) if m.sum() else np.nan)
    S["rho_dE_vs_M01_n8"], S["sign_agree_vs_M01"] = rho, sign

# ---- Ehull extension: rank correlation + survivor overlap on the top-N set (in-model hulls) ----
EH = {}
for key in MODELS:
    fp = f"{RUNS}/{key}/results/bench_ehull.csv"
    if os.path.exists(fp):
        EH[key] = pd.read_csv(fp).set_index("comp_id")
if EH and BASELINE in EH:
    e0 = EH[BASELINE]
    surv0 = set(e0.index[(e0.Ehull_eff < W.EHULL_EFF_CUT) & (e0.dE_pris > W.DELTA_FLOOR_MEV_FU)]) \
        if "dE_pris" in e0.columns else set()
    for key, e in EH.items():
        common = e0.index.intersection(e.index)
        S.loc[key, "n_ehull"] = len(common)
        if len(common) >= 5:
            S.loc[key, "rho_ehull_eff"] = spearmanr(e0.loc[common, "Ehull_eff"],
                                                    e.loc[common, "Ehull_eff"]).statistic
            if "dE_pris" in e.columns and "dE_pris" in e0.columns:
                S.loc[key, "rho_dE_pris_topN"] = spearmanr(e0.loc[common, "dE_pris"],
                                                           e.loc[common, "dE_pris"]).statistic
                S.loc[key, "sign_dE_pris_topN"] = float((np.sign(e0.loc[common, "dE_pris"])
                                                         == np.sign(e.loc[common, "dE_pris"])).mean())
        if "dE_pris" in e.columns:
            surv = set(e.index[(e.Ehull_eff < W.EHULL_EFF_CUT) & (e.dE_pris > W.DELTA_FLOOR_MEV_FU)])
            S.loc[key, "n_surv_topN"] = len(surv)
            S.loc[key, "surv_overlap_M01"] = (len(surv & surv0) / len(surv0)) if surv0 else np.nan
elif EH:
    print(f"[ehull] baseline {BASELINE} has no bench_ehull.csv -> overlap columns skipped")

# ---- top-N 4-point cross-model comparison: does model M reproduce the ORB top-N? ----
TG = {}
for key in MODELS:
    fp = f"{RUNS}/{key}/results/targets_x4.csv"
    if os.path.exists(fp):
        TG[key] = pd.read_csv(fp).set_index("comp_id")
if TG and BASELINE in TG:
    t0v = TG[BASELINE]
    for key, t in TG.items():
        common = t0v.index.intersection(t.index)
        S.loc[key, "n_targets_x4"] = len(common)
        if len(common) >= 3 and "ddE" in t.columns and "ddE" in t0v.columns:
            S.loc[key, "rho_ddE_topN"] = spearmanr(t0v.loc[common, "ddE"],
                                                   t.loc[common, "ddE"]).statistic
        for s in ("desod", "x067"):
            c = f"verdict_{s}"
            if c in t.columns and c in t0v.columns:
                S.loc[key, f"agree_{c}"] = float((t.loc[common, c] == t0v.loc[common, c]).mean())
        for c in ("dE_desod", "dE_x067"):
            if c in t.columns and c in t0v.columns:
                S.loc[key, f"MAE_{c}_vs_M01"] = float((t.loc[common, c] - t0v.loc[common, c]).abs().mean())
elif TG:
    print(f"[targets] baseline {BASELINE} has no targets_x4.csv -> comparison columns skipped")

S.to_csv(f"{BENCH}/results/bench_summary.csv")
GATES = [c for c in S.columns if c.startswith("gate_")]
print("=== gate ladder (True=pass, None=not computable) ===")
print(S[["model_id", "status"] + GATES].to_string())
print("\n=== dE per x, meV/f.u. (dE>0 = O3 승리) ===")
show = [f"{g}_{c}" for g in ("HOST", "HEO") for c in DE_COLS]
print(S[[c for c in show if c in S.columns]].round(1).to_string())
geo = [c for c in S.columns if any(c.endswith(suf) for suf in
       ("dV_pct_desod", "dA_pct_desod", "dh_perp_pct_desod", "vm_strain_desod"))]
if geo:
    print("\n=== geometry at x=0.52 (change-A decomposition, %; vm_strain dimensionless) ===")
    print(S[geo].round(3).to_string())
tg_cols = [c for c in ("n_targets_x4", "rho_ddE_topN", "agree_verdict_desod", "agree_verdict_x067",
                       "MAE_dE_desod_vs_M01", "MAE_dE_x067_vs_M01") if c in S.columns]
if tg_cols:
    print("\n=== top-N 4-point vs baseline (rho_ddE = 프로덕션 채점 성분의 순위 재현) ===")
    print(S[tg_cols].round(3).to_string())
eh_cols = [c for c in ("n_ehull", "rho_ehull_eff", "rho_dE_pris_topN", "sign_dE_pris_topN",
                       "n_surv_topN", "surv_overlap_M01") if c in S.columns]
if eh_cols:
    print("\n=== Ehull extension (in-model hulls, top-N pristine; MP-currency 모델끼리만 정식 비교) ===")
    print(S[eh_cols].round(3).to_string())
print("\n=== diagnostics / speed ===")
diag = [c for c in ("gap_pris", "margin_pris", "gap_x067", "gap_desod", "E_Na",
                    "rho_dE_vs_M01_n8", "sign_agree_vs_M01", "t_relax_median_s", "conv_rate",
                    "xrd_dir_ok_003", "xrd_dir_ok_110", "xrd_letter_agree") if c in S.columns]
print(S[diag].round(3).to_string())
print(f"\n-> {BENCH}/results/bench_summary.csv")


## 셀 8 — orb 일관성: ASE 경로 ↔ v4 torch-sim 경로

같은 모델(orb_mpa)·같은 시작구조에서 옵티마이저 경로만 다를 때 dE가 얼마나 움직이는지.
이 차이가 작아야(수 meV/f.u.) "벤치마크의 ASE 균일 경로가 프로덕션과 호환"이라는 각주가 성립한다.

In [ ]:
# ============ CELL 8: orb ASE vs torch-sim (v4 production) consistency ============
fp_v4 = os.environ["HEO_V4_RESULTS"]; fp_b = f"{RUNS}/orb_mpa/results/anchors_x4.csv"
if os.path.exists(fp_v4) and os.path.exists(fp_b):
    v4 = pd.read_csv(fp_v4, index_col=0); ba = pd.read_csv(fp_b, index_col=0)
    common = [c for c in DE_COLS if c in v4.columns and c in ba.columns]
    grps = [g for g in ("HOST", "HEO") if g in v4.index and g in ba.index]
    cmp_rows = []
    for g in grps:
        for c in common:
            cmp_rows.append(dict(group=g, metric=c, torchsim_v4=float(v4.loc[g, c]),
                                 ase_bench=float(ba.loc[g, c]),
                                 diff=float(ba.loc[g, c] - v4.loc[g, c])))
    CMP = pd.DataFrame(cmp_rows)
    CMP.to_csv(f"{BENCH}/results/orb_path_consistency.csv", index=False)
    print(CMP.round(2).to_string(index=False))
    dmax = CMP["diff"].abs().max()
    print(f"\nmax |dE(ASE) - dE(torch-sim)| = {dmax:.1f} meV/f.u. "
          + ("-> paths compatible (< delta floor 5)" if dmax < 5 else
         "-> PATH DIFFERENCE MATTERS: report it next to every cross-model comparison"))
else:
    print(f"skipped: need {fp_v4} (v4 run) and {fp_b} (orb_mpa bench run)")


## 셀 8b — XRD on 기존 v4 결과 (XRD_SPEC §5–§7)

기존 스크리닝 결과(heo_v2/heo_v4의 이완 CIF·cell 컬럼)에 시뮬레이션 XRD를 **후처리로만** 적용.
순서(§6): HOST 파일럿 → 통과 시 앵커+top-30 → (HEO_XRD_SCOPE=top500이면) top-500.
파일럿 실패 시 여기서 멈추고 보고한다(§5) — top-500 확장 금지.

In [ ]:
# ============ CELL 8b: XRD on the existing v4 run ============
import warnings
import xrd_tools, importlib; importlib.reload(xrd_tools)
V2, V4 = os.environ["HEO_V2_WORKDIR"], os.environ["HEO_V4_WORKDIR"]
SCOPE = os.environ["HEO_XRD_SCOPE"]; OUT = f"{BENCH}/xrd_v4"; os.makedirs(OUT, exist_ok=True)

fps = sorted(glob.glob(f"{V2}/checkpoints/energies*.csv")) + sorted(glob.glob(f"{V4}/checkpoints/energies*.csv"))
assert fps, f"no checkpoints under {V2} / {V4} -> mount the screening volumes"
EDF = pd.concat([pd.read_csv(f) for f in fps], ignore_index=True).drop_duplicates(["comp_id", "tag"])
if "cell" not in EDF.columns: EDF["cell"] = float("nan")
print(f"{len(EDF)} energy rows from {len(fps)} checkpoint files")

# composition lists + TM counts
counts_of = {aid: dict(c) for aid, c in W.ANCHORS.items()}
comp_ids = list(W.ANCHORS)
if SCOPE in ("top30", "top500"):
    n_top = 30 if SCOPE == "top30" else 500
    br = pd.read_csv(os.environ["HEO_BASE_RESULTS"], index_col="comp_id").dropna(subset=["dE_desod"])
    if {"score", "tier"} <= set(br.columns): br = br.sort_values(["tier", "score"])
    top = list(br.index[:n_top])
    cm = None
    for cand in (f"{V4}/results/compositions_master.csv", f"{V2}/results/compositions_master.csv"):
        if os.path.exists(cand): cm = pd.read_csv(cand).set_index("comp_id"); break
    assert cm is not None, "compositions_master.csv not found -> cannot map comp_id to site counts"
    for cid in top:
        if cid in cm.index:
            counts_of[cid] = json.loads(cm.loc[cid, "site_counts"]); comp_ids.append(cid)
    print(f"scope={SCOPE}: anchors + {len(comp_ids) - len(W.ANCHORS)} screening comps")

recs_of = W.recs_by_comp(EDF, comp_ids)
def loader(comp_id, tag):
    for d in (f"{V4}/structures/relaxed", f"{V2}/structures/relaxed"):
        fp = f"{d}/{comp_id}__{tag}.cif"
        if os.path.exists(fp):
            from pymatgen.core import Structure
            with warnings.catch_warnings():
                warnings.simplefilter("ignore"); return Structure.from_file(fp)
    return None

# --- stage 1: HOST pilot (spec section 5) ---
host_ids = [a for a in W.ANCHORS if a.startswith("HOST")]
xdf_pilot, refs = xrd_tools.run_xrd(host_ids, counts_of, recs_of, loader, OUT, W.X_STEMS, W.X_OF,
                                    n_vac=W.N_VAC_SAMPLES, tm_set=set(W.OXI))
assert len(xdf_pilot), ("HOST pilot found no relaxed CIFs -- check that structures/relaxed exists under "
                        f"{V4} or {V2} (HEO_SAVE_RELAXED=1 in the screening run)")
pilot = xrd_tools.host_direction_test(xdf_pilot)
print("\nHOST pilot:", {k: v for k, v in pilot.items() if not k.endswith("track")})
print("  tt_003 track (x desc):", {k: round(v, 3) for k, v in pilot["tt_003_track"].items()})
print("  tt_110 track (x desc):", {k: round(v, 3) for k, v in pilot["tt_110_track"].items()})
bad = int((xdf_pilot.chk_d003_resid > xrd_tools.D003_TOL).sum())
print(f"  d003 audit: {bad} rows over {xrd_tools.D003_TOL} A "
      f"(max resid {np.nanmax(xdf_pilot.chk_d003_resid):.4f} A)")
PILOT_OK = bool(pilot["xrd_dir_ok_003"]) and bool(pilot["xrd_dir_ok_110"]) and bad == 0
if not PILOT_OK:
    print("\nHOST PILOT FAILED -> stopping here per XRD_SPEC section 5 (no top-N expansion). "
          "Check the listed rows before rerunning.")

# --- stage 2: full scope ---
XDF = xdf_pilot
if PILOT_OK and len(comp_ids) > len(host_ids):
    XDF, _ = xrd_tools.run_xrd(comp_ids, counts_of, recs_of, loader, OUT, W.X_STEMS, W.X_OF,
                               n_vac=W.N_VAC_SAMPLES, tm_set=set(W.OXI))
XDF.to_csv(f"{OUT}/results_xrd_v4_long.csv", index=False)
wide = XDF.pivot_table(index="comp_id", columns=["stem", "phase"],
                       values=["tt_003", "tt_110", "d_003", "d_110", "a_eff", "c_eff",
                               "chk_d003_resid", "I_104_rel", "xrd_sim_O3", "xrd_sim_P3"],
                       aggfunc="first")
wide.columns = [f"{v}_{W.X_LABEL[s]}_{p}" for v, s, p in wide.columns]
wide.to_csv(f"{OUT}/results_xrd_v4.csv")
agree = XDF.xrd_agree_letter.dropna()
print(f"\n{len(XDF)} XRD rows -> {OUT}/results_xrd_v4.csv | "
      f"letter-classifier agreement {agree.mean()*100:.1f}% (n={len(agree)})" if len(agree)
      else f"\n{len(XDF)} XRD rows -> {OUT}/results_xrd_v4.csv")


## 셀 8c — XRD 워터폴 pptx (앵커 2종 + top-5, 편집 가능 네이티브 차트)

In [ ]:
# ============ CELL 8c: XRD waterfall pptx (editable native XY charts) ============
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.chart.data import XyChartData
from pptx.enum.chart import XL_CHART_TYPE

def _npz(comp_id, stem, phase):
    d = f"{OUT}/xrd/{comp_id}"
    for suf in ("kbest", "single"):
        fp = f"{d}/{stem}_{phase}_{suf}.npz"
        if os.path.exists(fp):
            z = np.load(fp); return z["tt"], z["I"]
    return None, None

wf_comps = ["HOST_v1", "HEO_v1"]
try:
    wf_comps += [c for c in top[:5]]           # top-5 from cell 8b's ranking (if it ran)
except NameError:
    pass
prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.33), Inches(7.5)
blank = prs.slide_layouts[6]
n_slides = 0
for comp in wf_comps:
    for phase in ("O3", "P3"):
        data = XyChartData(); found = False
        for j, stem in enumerate(W.X_GRID_DESC):
            tt, I = _npz(comp, stem, phase)
            if tt is None: continue
            found = True
            sr = data.add_series(f"x={W.X_OF[stem]:.2f}")
            for k in range(0, len(tt), 10):    # 0.1-degree sampling keeps the chart editable AND light
                sr.add_data_point(float(tt[k]), float(I[k] + 110.0 * j))   # waterfall offset
        if not found: continue
        slide = prs.slides.add_slide(blank)
        tb = slide.shapes.add_textbox(Inches(0.4), Inches(0.15), Inches(12), Inches(0.5))
        tb.text_frame.text = f"{comp} - {phase} simulated XRD waterfall (CuKa, offset 110/step, x desc)"
        tb.text_frame.paragraphs[0].font.size = Pt(18)
        slide.shapes.add_chart(XL_CHART_TYPE.XY_SCATTER_LINES_NO_MARKERS,
                               Inches(0.4), Inches(0.8), Inches(12.5), Inches(6.4), data)
        n_slides += 1
fp_pptx = f"{BENCH}/results/xrd_waterfall.pptx"
prs.save(fp_pptx)
print(f"{n_slides} slides -> {fp_pptx}")


## 셀 9 — 리포트 `results/benchmark_report_stage0.md`

In [ ]:
# ============ CELL 9: stage-0 report ============
L = []
L.append("# uMLIP 12-model benchmark — stage 0 (DFT 없음: 관문 0·1 + 상호 비교 + XRD)\n")
L.append(f"실행: {time.strftime('%Y-%m-%d %H:%M')} | 모델 {len(MODELS)}종 | 앵커 HOST/HEO 4-point | "
         f"엔진 ASE FIRE + FrechetCellFilter (fmax {W.FMAX}, max_steps {W.MAX_STEPS})\n")
L.append("## 관문 사다리 (스펙 §5)\n")
L.append("| 단계 | 이번 실행 | 비고 |\n|---|---|---|")
L.append("| 관문 0 화폐 | models_registry.yml | 비 MP 화폐는 대조군 (§10-6) |")
L.append("| 관문 1 앵커 방향 | 아래 gate_* 컬럼 | HOST 전이·V창·프리스틴 방향 |")
L.append("| 관문 2 부호 / 지표 3-4 | **DFT 대기** | CORE/FRAMES 도착 후 |")
L.append("| 지표 5 실무 | t_relax_median_s, conv_rate | |\n")
L.append("## 결과 표\n")
try:
    L.append(S.reset_index().to_markdown(index=False))
except Exception:
    L.append("```\n" + S.to_string() + "\n```")
L.append("\n## 해석 규약\n- dE = [E(P3)-E(O3)]/27 meV/f.u., **dE>0 = O3 승리**. gap_* = HEO-HOST (클수록 지연 방향).")
L.append("- 우열 선언 금지: n=8 상호 비교는 참고치. 짝지은 부트스트랩 CI는 DFT 정답지 도착 후 (§5).")
L.append("- ASE 경로: 프로덕션 v4(torch-sim)와의 차이는 orb_path_consistency.csv 참조.")
L.append("\n## XRD 한계 (XRD_SPEC §8)\n- 0 K 구조, Debye-Waller 없음 → 고각 세기 과대: 세기는 상대 비교만.")
L.append("- FWHM 0.15°는 설정값(물리 아님). 실험 피크 폭과 비교 금지.")
L.append("- 점유율 평균은 SRO를 지움(완전 무작위 가정) — PDF 영역은 다루지 않음.")
fp_rep = f"{BENCH}/results/benchmark_report_stage0.md"
open(fp_rep, "w").write("\n".join(L))
Path(BENCH, "results", "DONE").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
print(f"-> {fp_rep}\nDONE sentinel written (watchdog: log-only under terminate=none)")


## RunPod에서 실행

1. **팟**: PyTorch 2.x CUDA 이미지, GPU ≥ 2 권장(12모델 직렬이면 1도 가능), Network Volume에
   `/workspace/heo_v2`(v3 결과)·`/workspace/heo_v4`(v4 결과) 마운트.
2. **업로드**: 이 노트북 + `heo_worker.py`(= heo_worker-3.py 이름 변경) + `bench_models.py` +
   `bench_anchor_runner.py` + `xrd_tools.py` + `models_registry.yml`을 같은 폴더에.
3. **env** (터미널에서 export 후 jupyter 시작, 키는 노트북에 쓰지 말 것):
   `HF_TOKEN`(eSEN gated 시), `HEO_EQUFLASH_CKPT`(Figshare 수동 다운로드 경로),
   `HEO_BENCH_MODELS`(스모크: `orb_mpa`), `HEO_XRD_SCOPE`(pilot|top30|top500).
4. **스모크 먼저**: `HEO_BENCH_MODELS=orb_mpa`로 Run All → 셀 8에서 v4 대비 |ΔdE| < 5 meV/f.u. 확인
   → 이후 `HEO_BENCH_MODELS=` 비우고 12종 전체.
5. **브라우저가 끊겨도 계산은 계속**: 안전하게는 터미널에서
   `nohup jupyter nbconvert --to notebook --execute --inplace heo_bench_runpod_v1.ipynb > nbrun.log 2>&1 &`
6. 끝나면 `results/bench_summary.csv`·`benchmark_report_stage0.md`·`xrd_waterfall.pptx` 회수,
   **팟 수동 종료** (워치독은 terminate=none이라 로그만 남긴다).

해석 방법은 저장소의 `BENCH_PIPELINE_REVIEW.md` 참조.